# Lesson 7: Gradient Descent

*ML Academy — companion notebook*

Train any model by walking downhill: learning rates, batch vs stochastic, and convergence.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-07.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python

In [1]:
import numpy as np

x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 4, 5, 4, 5], dtype=float)

w, b = 0.0, 0.0
lr = 0.05
for step in range(1000):
    y_hat = w * x + b
    error = y_hat - y
    grad_w = 2 * np.mean(error * x)     # dL/dw
    grad_b = 2 * np.mean(error)         # dL/db
    w -= lr * grad_w
    b -= lr * grad_b
    if step in (0, 9, 99, 999):
        print(step + 1, round(w, 3), round(b, 3), round(np.mean((w * x + b - y) ** 2), 3))

1 1.32 0.4 1.646
10 1.041 0.607 0.942
100 0.695 1.857 0.501
1000 0.6 2.2 0.48


In [2]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = make_pipeline(StandardScaler(),
                      SGDRegressor(learning_rate="invscaling", eta0=0.01, max_iter=1000, random_state=0))
model.fit(X_train, y_train)
print(round(model.score(X_test, y_test), 3))     # R² on the test set

0.456


## 14. Going deeper: modern optimisers and learning-rate schedules

In [3]:
import numpy as np

def run(update, grad, start, steps=20000, tol=1e-3):
    """Steps until w is within tol of the minimum at (0, 0)."""
    w, state = np.array(start, dtype=float), {}
    for k in range(1, steps + 1):
        w = update(w, grad(w), state, k)
        if np.linalg.norm(w) < tol: return k
    return None

def plain(lr):
    return lambda w, g, s, k: w - lr * g

def momentum(lr, beta):
    def step(w, g, s, k):
        s["v"] = beta * s.get("v", 0) + g                  # a running "velocity"
        return w - lr * s["v"]
    return step

def adam(lr, b1=0.9, b2=0.999, eps=1e-8):
    def step(w, g, s, k):
        s["m"] = b1 * s.get("m", 0) + (1 - b1) * g         # average gradient
        s["v"] = b2 * s.get("v", 0) + (1 - b2) * g**2      # average squared gradient
        m_hat, v_hat = s["m"] / (1 - b1**k), s["v"] / (1 - b2**k)   # bias correction
        return w - lr * m_hat / (np.sqrt(v_hat) + eps)
    return step

valleys = {"narrow (curvatures 2 and 20)":     (lambda w: np.array([2 * w[0], 20 * w[1]]), 20),
           "very narrow (curvatures 2 and 2000)": (lambda w: np.array([2 * w[0], 2000 * w[1]]), 2000)}
for name, (g, top) in valleys.items():
    lr = 1.8 / top                                         # just under the stability limit 2 / top
    print(name)
    print(f"  plain gradient descent: {run(plain(lr), g, [10, 1])} steps")
    print(f"  momentum (beta 0.3):    {run(momentum(lr * 1.2, 0.3), g, [10, 1])} steps")
    print(f"  Adam (lr 0.1):          {run(adam(0.1), g, [10, 1])} steps")

narrow (curvatures 2 and 20)
  plain gradient descent: 47 steps
  momentum (beta 0.3):    17 steps
  Adam (lr 0.1):          310 steps
very narrow (curvatures 2 and 2000)
  plain gradient descent: 5113 steps
  momentum (beta 0.3):    2977 steps
  Adam (lr 0.1):          310 steps


In [4]:
from sklearn.datasets import load_diabetes

# Mini-batch SGD on the diabetes data: constant learning rate vs a decaying schedule
Xd, yd = load_diabetes(return_X_y=True)
Xd = (Xd - Xd.mean(0)) / Xd.std(0); yd = (yd - yd.mean()) / yd.std()
w_best = np.linalg.lstsq(Xd, yd, rcond=None)[0]
best = np.mean((Xd @ w_best - yd) ** 2)

def sgd(schedule, epochs=60, batch=16, seed=0):
    rng = np.random.default_rng(seed); w = np.zeros(10); step = 0; tail = []
    for ep in range(epochs):
        for idx in np.array_split(rng.permutation(len(yd)), len(yd) // batch):
            g = 2 * Xd[idx].T @ (Xd[idx] @ w - yd[idx]) / len(idx)
            w -= schedule(step) * g; step += 1
        if ep >= epochs - 10: tail.append(np.mean((Xd @ w - yd) ** 2))
    return np.mean(tail) - best, np.std(tail)

for name, sch in [("constant 0.05", lambda t: 0.05),
                  ("decay 0.05/(1 + t/200)", lambda t: 0.05 / (1 + t / 200))]:
    gap, wobble = sgd(sch)
    print(f"{name:24}: extra loss above the best {gap:.4f}, wobble between epochs {wobble:.4f}")

constant 0.05           : extra loss above the best 0.0074, wobble between epochs 0.0047
decay 0.05/(1 + t/200)  : extra loss above the best 0.0019, wobble between epochs 0.0001


In [5]:
# Gradient clipping: cap the gradient's length before stepping
def clip(g, max_norm=1.0):
    n = np.linalg.norm(g)
    return g * (max_norm / n) if n > max_norm else g

huge = np.array([300.0, -400.0])                 # an exploding gradient, length 500
print("before clipping:", huge, " length", np.linalg.norm(huge))
print("after clipping: ", clip(huge), " length", np.linalg.norm(clip(huge)))

before clipping: [ 300. -400.]  length 500.0
after clipping:  [ 0.6 -0.8]  length 1.0


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
x = df["size_m2"].to_numpy(); y = df["price_lakh"].to_numpy()
xs = (x - x.mean()) / x.std()            # standardise so one learning rate suits both parameters
w, b, lr = 0.0, 0.0, 0.1
for step in range(200):
    err = w * xs + b - y
    w -= lr * 2 * np.mean(err * xs)
    b -= lr * 2 * np.mean(err)
print("in standardised units: w =", round(w, 3), " b =", round(b, 3))
print("back in m2: slope =", round(w / x.std(), 3), " intercept =", round(b - w * x.mean() / x.std(), 2))

in standardised units: w = 7.482  b = 72.083
back in m2: slope = 0.29  intercept = 41.11


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: The gradient of the MSE

Compute ∂L/∂w for the model ŷ = wx + b at w = 0, b = 0.

In [8]:
import numpy as np
x = np.array([1, 2, 3, 4, 5.0]); y = np.array([2, 4, 5, 4, 5.0])
w, b = 0.0, 0.0
err = w * x + b - y
grad_w = None  # ← your code here
check(grad_w, -26.4, hint="2 × mean(error × x). The lesson's worked example gives −26.4.")

❌ Not yet. Hint: 2 × mean(error × x). The lesson's worked example gives −26.4.


### Challenge 2: The update rule

Complete the loop so it runs 1000 steps of gradient descent. w should end near 0.6.

In [9]:
import numpy as np
x = np.array([1, 2, 3, 4, 5.0]); y = np.array([2, 4, 5, 4, 5.0])
w, b, lr = 0.0, 0.0, 0.05
for step in range(1000):
    err = w * x + b - y
    grad_w, grad_b = 2 * np.mean(err * x), 2 * np.mean(err)
    pass  # ← your code here: update w
    b = b - lr * grad_b
w_final = w
check(w_final, 0.6000000206173624, hint='Mirror the line for b: w = w - lr * grad_w.')

❌ Not yet. Hint: Mirror the line for b: w = w - lr * grad_w.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. The gradient of the MSE** — the missing line:

```python
grad_w = 2 * np.mean(err * x)
```

**2. The update rule** — the missing line:

```python
w = w - lr * grad_w
```